# LiDAR, Point Clouds and Camera–LiDAR Fusion — notebook
Code from the tutorial PDF, in order. Run top to bottom. Needs `numpy scipy scikit-learn matplotlib` (`%pip install numpy scipy scikit-learn matplotlib`).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

## 1. Synthetic street and ray caster (Section 2)

In [ ]:
import numpy as np

# ---------- a synthetic street, in the LiDAR frame (x forward, y left, z up, metres) ----------
SENSOR_HEIGHT, SLOPE = 1.73, 0.02            # road rises 2 cm per metre ahead
def ground_z(x):
    return -SENSOR_HEIGHT + SLOPE * x

# label ids: 1 ground, 2 wall, 3 pole, 10+ cars, 20+ pedestrians
BOXES = [  # (label, x_min, x_max, y_min, y_max, height)
    (10, 10.0, 14.5, -2.8, -1.0, 1.5), (11, 23.0, 27.5, 1.2, 3.0, 1.6),
    (12, 38.0, 42.6, -2.6, -0.7, 1.5), (2, 0.0, 60.0, 8.0, 8.4, 6.0)]
CYLINDERS = [  # (label, cx, cy, radius, height)
    (20, 9.0, 3.5, 0.3, 1.75), (21, 34.0, -4.5, 0.3, 1.75), (22, 17.0, 7.4, 0.3, 1.8),
    (3, 15.0, 4.6, 0.15, 4.0)]

def ray_cast(origin, dirs, max_range=100.0):
    """First hit of every ray: returns distance t (inf = no hit) and label."""
    o = np.asarray(origin, float)
    t_best = np.full(len(dirs), np.inf); lab = np.zeros(len(dirs), int)
    # ground plane z - SLOPE x = -SENSOR_HEIGHT
    den = dirs[:, 2] - SLOPE * dirs[:, 0]
    with np.errstate(divide="ignore", invalid="ignore"):
        t = (-SENSOR_HEIGHT - o[2] + SLOPE * o[0]) / den
    hit = (t > 0) & (t < t_best); t_best[hit], lab[hit] = t[hit], 1
    for label, x0, x1, y0, y1, h in BOXES:               # slab method for boxes
        z0 = ground_z((x0 + x1) / 2); lo = np.array([x0, y0, z0]); hi = np.array([x1, y1, z0 + h])
        with np.errstate(divide="ignore", invalid="ignore"):
            ta, tb = (lo - o) / dirs, (hi - o) / dirs
        tn = np.nanmax(np.minimum(ta, tb), axis=1); tf = np.nanmin(np.maximum(ta, tb), axis=1)
        hit = (tn <= tf) & (tn > 0) & (tn < t_best); t_best[hit], lab[hit] = tn[hit], label
    for label, cx, cy, r, h in CYLINDERS:                # vertical cylinders
        dx, dy = dirs[:, 0], dirs[:, 1]; ox, oy = o[0] - cx, o[1] - cy
        a = dx**2 + dy**2; b = 2 * (ox * dx + oy * dy); c = ox**2 + oy**2 - r**2
        disc = b**2 - 4 * a * c
        with np.errstate(invalid="ignore", divide="ignore"):
            t = (-b - np.sqrt(disc)) / (2 * a)
        z = o[2] + t * dirs[:, 2]; zb = ground_z(cx)
        hit = (disc > 0) & (t > 0) & (z > zb) & (z < zb + h) & (t < t_best)
        t_best[hit], lab[hit] = t[hit], label
    t_best[t_best > max_range] = np.inf
    return t_best, lab

## 2. Simulate one 64-beam scan

In [ ]:
# ---------- a 64-beam rotating LiDAR (geometry similar to the Velodyne HDL-64E in KITTI) ----------
rng = np.random.default_rng(0)
elev = np.radians(np.linspace(2.0, -24.8, 64))          # 64 beams, 0.43 deg apart
azim = np.radians(np.arange(0, 360, 0.2))               # 0.2 deg horizontal step
E, A = np.meshgrid(elev, azim, indexing="ij")
dirs = np.c_[(np.cos(E) * np.cos(A)).ravel(), (np.cos(E) * np.sin(A)).ravel(), np.sin(E).ravel()]

rng_m, labels = ray_cast(np.zeros(3), dirs)
ok = np.isfinite(rng_m) & (rng.random(len(rng_m)) > 0.02)      # 2 % dropouts
rng_m = rng_m + rng.normal(0, 0.02, len(rng_m))                 # 2 cm range noise
points = dirs[ok] * rng_m[ok, None]
labels = labels[ok]
intensity = np.clip(np.where(labels == 1, 0.2, 0.5) + rng.normal(0, 0.05, len(points)), 0, 1)
cloud = np.c_[points, intensity].astype(np.float32)               # x, y, z, intensity

print(f"{len(dirs)} beams fired, {len(cloud)} points returned")
for name, lab in [("pedestrian at  9 m", 20), ("pedestrian at 34 m", 21), ("car at 10 m", 10),
                  ("car at 38 m", 12)]:
    print(f"{name}: {np.sum(labels == lab):5d} points")

In [ ]:
m = labels > 0
plt.figure(figsize=(8, 8))
plt.scatter(points[:, 0], points[:, 1], s=0.2, c=labels, cmap="tab20")
plt.xlim(-30, 60); plt.ylim(-25, 25); plt.gca().set_aspect("equal")
plt.title("bird's-eye view"); plt.show()

## 3. Points on target (Section 3)

In [ ]:
def points_on_target(height, width, distance, v_res_deg=0.43, h_res_deg=0.2):
    """Rough number of LiDAR points on an object: beams x columns that hit it."""
    rows = height / (distance * np.radians(v_res_deg))
    cols = width / (distance * np.radians(h_res_deg))
    return rows, cols, rows * cols

for d in [9, 34, 60]:
    rows, cols, n = points_on_target(1.75, 0.6, d)
    print(f"pedestrian at {d:2d} m: ~{rows:4.1f} beams x {cols:4.1f} columns = ~{n:5.0f} points")

## 4. Crop and voxel grid (Section 4)

In [ ]:
def crop(cloud, x=(0, 50), y=(-10, 10), z=(-3, 3)):
    p = cloud[:, :3]
    m = (p[:, 0] > x[0]) & (p[:, 0] < x[1]) & (p[:, 1] > y[0]) & (p[:, 1] < y[1]) \
        & (p[:, 2] > z[0]) & (p[:, 2] < z[1])
    return cloud[m], m

def voxel_downsample(cloud, voxel=0.2):
    """Replace all points in each voxel by their centroid."""
    idx = np.floor(cloud[:, :3] / voxel).astype(np.int64)
    _, inverse, counts = np.unique(idx, axis=0, return_inverse=True, return_counts=True)
    inverse = inverse.ravel()
    out = np.zeros((len(counts), cloud.shape[1]))
    np.add.at(out, inverse, cloud)
    return out / counts[:, None], inverse

roi, roi_mask = crop(cloud)
roi_labels = labels[roi_mask]
vox, vox_inv = voxel_downsample(roi, 0.2)
print(f"region of interest: {len(roi)} points;  after 0.2 m voxel grid: {len(vox)} points "
      f"({len(vox) / len(roi):.0%})")

## 5. Ground removal: threshold vs RANSAC (Section 5)

In [ ]:
def ransac_plane(p, threshold=0.15, iterations=200, seed=0):
    """Fit the dominant plane n.p + d = 0 with RANSAC (3-point samples), then refit."""
    r = np.random.default_rng(seed)
    best = np.zeros(len(p), bool)
    for _ in range(iterations):
        a, b, c = p[r.choice(len(p), 3, replace=False)]
        n = np.cross(b - a, c - a)
        if np.linalg.norm(n) < 1e-6:
            continue
        n /= np.linalg.norm(n)
        if abs(n[2]) < 0.9:                       # only accept roughly horizontal planes
            continue
        inl = np.abs((p - a) @ n) < threshold
        if inl.sum() > best.sum():
            best = inl
    # least-squares refit on the inliers: smallest singular vector of the centred points
    q = p[best]; c = q.mean(0)
    n = np.linalg.svd(q - c, full_matrices=False)[2][-1]
    return n, -n @ c, np.abs((p - c) @ n) < threshold

p = roi[:, :3]
is_ground_true = roi_labels == 1
n, d, ground_ransac = ransac_plane(p)
ground_naive = p[:, 2] < -SENSOR_HEIGHT + 0.2           # "everything low is ground"

def report(name, pred):
    tp = np.sum(pred & is_ground_true)
    precision, recall = tp / pred.sum(), tp / is_ground_true.sum()
    lost = np.sum(pred & ~is_ground_true)                # obstacle points wrongly removed
    print(f"{name:22s} ground recall {recall:6.1%}, precision {precision:6.1%}, "
          f"obstacle points removed {lost}")

report("height threshold", ground_naive)
report("RANSAC plane", ground_ransac)
print(f"fitted plane normal {n.round(3)}, slope {-n[0] / n[2]:.3f} (true {SLOPE})")

## 6. KD-tree clustering and DBSCAN (Section 6)

In [ ]:
from scipy.spatial import cKDTree
from sklearn.cluster import DBSCAN

obst = p[~ground_ransac]
obst_true = roi_labels[~ground_ransac]

def euclidean_clusters(pts, radius=0.6, min_size=8):
    """Region growing with a KD-tree: neighbours closer than radius join the same cluster."""
    tree = cKDTree(pts)
    lab = np.full(len(pts), -1)
    cid = 0
    for seed in range(len(pts)):
        if lab[seed] != -1:
            continue
        queue, members = [seed], []
        lab[seed] = cid
        while queue:
            i = queue.pop()
            members.append(i)
            for j in tree.query_ball_point(pts[i], radius):
                if lab[j] == -1:
                    lab[j] = cid; queue.append(j)
        if len(members) < min_size:
            lab[members] = -2                      # too small: noise
        else:
            cid += 1
    return lab

lab_kd = euclidean_clusters(obst)
lab_db = DBSCAN(eps=0.6, min_samples=5).fit_predict(obst)

def cluster_report(name, lab):
    ids = [c for c in np.unique(lab) if c >= 0]
    pure = [np.bincount(obst_true[lab == c]).max() / np.sum(lab == c) for c in ids]
    found = {np.bincount(obst_true[lab == c]).argmax() for c in ids}
    objects = {l for l in np.unique(obst_true) if l >= 2}
    print(f"{name:20s} {len(ids):2d} clusters, mean purity {np.mean(pure):.1%}, "
          f"objects found {len(found & objects)} of {len(objects)}")

cluster_report("KD-tree clustering", lab_kd)
cluster_report("DBSCAN", lab_db)

## 7. Boxes and distances (Section 7)

In [ ]:
def cluster_box(pts):
    """Axis-aligned box and the distance from the sensor to the closest point."""
    lo, hi = pts.min(0), pts.max(0)
    closest = np.min(np.linalg.norm(pts[:, :2], axis=1))
    return lo, hi, closest

names = {10: "car A", 11: "car B", 12: "car C", 20: "pedestrian 1", 21: "pedestrian 2",
         22: "pedestrian 3", 3: "pole", 2: "wall"}
rows = []
for c in [c for c in np.unique(lab_kd) if c >= 0]:
    pts = obst[lab_kd == c]
    if len(pts) < 30:
        continue                                   # small fragments (see text)
    true = np.bincount(obst_true[lab_kd == c]).argmax()
    lo, hi, closest = cluster_box(pts)
    rows.append((closest, names.get(true, true), len(pts), hi - lo))
for closest, name, npts, size in sorted(rows):
    print(f"{name:13s} {npts:5d} pts  size {size[0]:4.1f} x {size[1]:4.1f} x {size[2]:3.1f} m  "
          f"closest {closest:5.1f} m")

## 8. Camera, extrinsics, rendered image, detector boxes (Section 8)

In [ ]:
# ---------- a camera next to the LiDAR (geometry similar to KITTI's colour camera) ----------
W_IMG, H_IMG = 1242, 375
K = np.array([[721.5, 0, 609.6], [0, 721.5, 172.9], [0, 0, 1]])
R_cam_lidar = np.array([[0, -1, 0], [0, 0, -1], [1, 0, 0]], float)   # body axes -> optical axes
cam_pos = np.array([0.27, 0.0, -0.08])                  # camera position in the LiDAR frame
t_cam_lidar = -R_cam_lidar @ cam_pos
T_cam_lidar = np.eye(4); T_cam_lidar[:3, :3], T_cam_lidar[:3, 3] = R_cam_lidar, t_cam_lidar

# render the camera image by ray casting the same scene (label image + depth)
uu, vv = np.meshgrid(np.arange(W_IMG), np.arange(H_IMG))
rays_cam = (np.linalg.inv(K) @ np.c_[uu.ravel(), vv.ravel(), np.ones(uu.size)].T).T
rays_lidar = rays_cam @ R_cam_lidar                                  # = (R^T d)^T
rays_lidar /= np.linalg.norm(rays_lidar, axis=1, keepdims=True)
t_img, lab_img = ray_cast(cam_pos, rays_lidar)
label_image = lab_img.reshape(H_IMG, W_IMG)

# "detector": a 2D box for every car and pedestrian visible in the image. Like a real detector it
# covers the whole object (also hidden parts) and is a little loose: projected 3D corners + 10 %.
def object_corners(lab):
    for L, x0, x1, y0, y1, h in BOXES:
        if L == lab:
            z0 = ground_z((x0 + x1) / 2)
            return np.array([[x, y, z] for x in (x0, x1) for y in (y0, y1) for z in (z0, z0 + h)])
    for L, cx, cy, r, h in CYLINDERS:
        if L == lab:
            z0 = ground_z(cx)
            return np.array([[cx + a, cy + b, z] for a in (-r, r) for b in (-r, r)
                             for z in (z0, z0 + h)])

detections = []
for lab in [10, 11, 12, 20, 21, 22]:
    if np.sum(label_image == lab) < 30:                     # not visible enough
        continue
    c = (T_cam_lidar @ np.c_[object_corners(lab), np.ones(8)].T)[:3]
    uv_c = (K @ c)[:2] / (K @ c)[2]
    lo, hi = uv_c.min(1), uv_c.max(1)
    margin = 0.1 * (hi - lo)
    detections.append((lab, np.r_[lo - margin, hi + margin] + rng.normal(0, 2, 4)))
print(f"camera sees {len(detections)} objects:", [names[l] for l, _ in detections])

## 9. LiDAR → image projection (Section 9)

In [ ]:
def project_lidar_to_image(pts_lidar, T_cam_lidar, K, width, height):
    """LiDAR points -> pixel coordinates and camera depth; keeps points in front of the camera."""
    p_h = np.c_[pts_lidar, np.ones(len(pts_lidar))]
    p_cam = (T_cam_lidar @ p_h.T).T[:, :3]                  # 1. into the camera frame
    front = p_cam[:, 2] > 0.5                               # 2. drop points behind the camera
    uvw = (K @ p_cam[front].T).T                            # 3. pinhole projection
    uv = uvw[:, :2] / uvw[:, 2:]
    inside = (uv[:, 0] >= 0) & (uv[:, 0] < width) & (uv[:, 1] >= 0) & (uv[:, 1] < height)
    idx = np.nonzero(front)[0][inside]
    return uv[inside], p_cam[front][inside, 2], idx         # pixels, depth, index into the cloud

uv, depth, idx = project_lidar_to_image(points, T_cam_lidar, K, W_IMG, H_IMG)
print(f"{len(points)} LiDAR points, {len(uv)} land inside the camera image "
      f"({len(uv) / len(points):.0%}: the camera sees only the front)")

# check: a projected point should hit the same object in the camera's label image
hit = label_image[uv[:, 1].astype(int), uv[:, 0].astype(int)]
print(f"projected points landing on the same object in the image: {np.mean(hit == labels[idx]):.1%}")

In [ ]:
plt.figure(figsize=(14, 5))
plt.imshow(label_image, cmap="tab20", alpha=0.25)
plt.scatter(uv[:, 0], uv[:, 1], s=1, c=depth, cmap="turbo", vmax=45)
plt.xlim(0, W_IMG); plt.ylim(H_IMG, 0); plt.colorbar(label="depth (m)"); plt.show()

## 10. Distance for each 2D box (Section 10)

In [ ]:
# give every camera detection a distance from the LiDAR points inside its box
ground_all = np.abs(points @ n + d) < 0.15                       # RANSAC plane on the full cloud
true_dist = {10: 10.0, 11: 23.0, 12: 38.0, 20: 8.7, 21: 33.7, 22: 16.7}   # closest surface (x)

# LiDAR cluster id for every point of the full cloud (-1 = ground, outside ROI, or noise)
cluster_id = np.full(len(points), -1)
obst_index = np.nonzero(roi_mask)[0][~ground_ransac]
cluster_id[obst_index] = lab_kd

def box_distance(box, method, calib=T_cam_lidar):
    uv_, depth_, idx_ = project_lidar_to_image(points, calib, K, W_IMG, H_IMG)
    u, v = uv_[:, 0], uv_[:, 1]
    inb = (u >= box[0]) & (u <= box[2]) & (v >= box[1]) & (v <= box[3])
    if method == "median":                                    # all points in the box
        z = depth_[inb]
        return np.median(z) + cam_pos[0] if len(z) else np.nan
    z = np.sort(depth_[inb & ~ground_all[idx_]])              # ground removed
    groups = np.split(z, np.nonzero(np.diff(z) > 1.0)[0] + 1)  # split depths at jumps > 1 m
    if method == "nearest group":
        return np.median(groups[0]) + cam_pos[0]
    if method == "largest group":
        return np.median(max(groups, key=len)) + cam_pos[0]
    # "best cluster": the LiDAR cluster whose projection lies most completely inside the box
    best, best_score = None, (0, 0)
    for c in np.unique(cluster_id[idx_[inb]]):
        if c < 0:
            continue
        inside = np.sum(cluster_id[idx_[inb]] == c)
        score = (inside / np.sum(cluster_id[idx_] == c), inside)
        if score > best_score:
            best, best_score = c, score
    return points[cluster_id == best, 0].min() if best is not None else np.nan

methods = ["median", "nearest group", "largest group", "best cluster"]
print(f"{'object':13s} {'true':>6s}" + "".join(f"{m:>15s}" for m in methods))
FUSION = {}
for lab, box in detections:
    est = [box_distance(box, m) for m in methods]
    FUSION[lab] = est
    print(f"{names[lab]:13s} {true_dist[lab]:6.1f}" + "".join(f"{e:15.2f}" for e in est))

## 11. Effect of calibration error (Section 11)

In [ ]:
def rotz(deg):
    a = np.radians(deg)
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1]])

def on_correct_object(T, lab_ids):
    """Fraction of LiDAR points of the given objects that project onto the same object."""
    uv_, _, idx_ = project_lidar_to_image(points, T, K, W_IMG, H_IMG)
    sel = np.isin(labels[idx_], lab_ids)
    seen = label_image[uv_[sel, 1].astype(int), uv_[sel, 0].astype(int)]
    return np.mean(seen == labels[idx_][sel])

print("yaw error   shift   car 10 m   car 38 m   ped 9 m   ped 17 m")
for err_deg in [0.0, 0.25, 0.5, 1.0, 2.0]:
    T_bad = T_cam_lidar.copy()
    T_bad[:3, :3] = R_cam_lidar @ rotz(err_deg)              # extrinsic yaw error
    shift = 721.5 * np.tan(np.radians(err_deg))               # pixels
    share = [on_correct_object(T_bad, [lab]) for lab in (10, 12, 20, 22)]
    print(f"{err_deg:5.2f} deg  {shift:4.1f} px " + "".join(f"{v:10.1%}" for v in share))

## 12. KITTI .bin and calib files (Section 13.1)

In [ ]:
import os, tempfile

def save_kitti_bin(path, cloud):
    cloud.astype(np.float32).tofile(path)                  # x, y, z, intensity as float32

def load_kitti_bin(path):
    return np.fromfile(path, dtype=np.float32).reshape(-1, 4)

def save_kitti_calib(path, K, T_cam_lidar):
    P2 = np.c_[K, np.zeros(3)]
    with open(path, "w") as f:
        f.write("P2: " + " ".join(f"{v:.6e}" for v in P2.ravel()) + "\n")
        f.write("R0_rect: " + " ".join(f"{v:.6e}" for v in np.eye(3).ravel()) + "\n")
        f.write("Tr_velo_to_cam: " + " ".join(f"{v:.6e}" for v in T_cam_lidar[:3].ravel()) + "\n")

def load_kitti_calib(path):
    """Read P2, R0_rect and Tr_velo_to_cam from a KITTI object-detection calib file."""
    vals = {}
    for line in open(path):
        if ":" in line:
            key, rest = line.split(":", 1)
            vals[key] = np.array([float(x) for x in rest.split()])
    P2 = vals["P2"].reshape(3, 4)
    R0 = np.eye(4); R0[:3, :3] = vals["R0_rect"].reshape(3, 3)
    Tr = np.eye(4); Tr[:3] = vals["Tr_velo_to_cam"].reshape(3, 4)
    return P2, R0, Tr

def project_kitti(pts, P2, R0, Tr):
    """KITTI convention: x_img = P2 * R0_rect * Tr_velo_to_cam * X_velo."""
    X = np.c_[pts, np.ones(len(pts))].T
    cam = R0 @ Tr @ X
    front = cam[2] > 0.5
    uvw = P2 @ cam[:, front]
    return (uvw[:2] / uvw[2]).T, cam[2, front]

tmp = tempfile.mkdtemp()
save_kitti_bin(os.path.join(tmp, "000000.bin"), cloud)
save_kitti_calib(os.path.join(tmp, "000000.txt"), K, T_cam_lidar)
c2 = load_kitti_bin(os.path.join(tmp, "000000.bin"))
P2, R0, Tr = load_kitti_calib(os.path.join(tmp, "000000.txt"))
uv_k, _ = project_kitti(c2[:, :3], P2, R0, Tr)
p_cam = (T_cam_lidar @ np.c_[points, np.ones(len(points))].T)[:3]
uvw = K @ p_cam[:, p_cam[2] > 0.5]
uv_ref = (uvw[:2] / uvw[2]).T
print(f"reloaded {len(c2)} points; KITTI-style projection agrees with ours: "
      f"max difference {np.abs(uv_k - uv_ref).max():.4f} px")

## 13. Open3D version (run in your WSL .venv after `pip install open3d`; uses `cloud` from above)
Not run while making the tutorial.

In [ ]:
# import numpy as np
# import open3d as o3d                     # pip install open3d   (Python 3.8 - 3.12)
# 
# pcd = o3d.geometry.PointCloud()
# pcd.points = o3d.utility.Vector3dVector(cloud[:, :3].astype(np.float64))
# 
# roi_box = o3d.geometry.AxisAlignedBoundingBox(min_bound=np.array([0.0, -10.0, -3.0]),
#                                               max_bound=np.array([50.0, 10.0, 3.0]))
# roi = pcd.crop(roi_box)
# down = roi.voxel_down_sample(voxel_size=0.2)
# 
# plane, inliers = down.segment_plane(distance_threshold=0.15, ransac_n=3, num_iterations=200)
# ground = down.select_by_index(inliers)
# obstacles = down.select_by_index(inliers, invert=True)
# 
# labels = np.array(obstacles.cluster_dbscan(eps=0.6, min_points=5))
# boxes = []
# for c in range(labels.max() + 1):
#     cluster = obstacles.select_by_index(np.nonzero(labels == c)[0])
#     box = cluster.get_axis_aligned_bounding_box()
#     box.color = (1, 0, 0)
#     boxes.append(box)
# 
# ground.paint_uniform_color([0.6, 0.6, 0.6])
# o3d.visualization.draw_geometries([ground, obstacles, *boxes])      # opens a 3D viewer window